# 🎵 Spotify 2023 — ETL & Análise Exploratória de Dados

**Dataset:** [Most Streamed Spotify Songs 2023](https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023) (Kaggle)  
**Stack:** Python · Pandas · NumPy · Plotly  
**Webapp:** [Dashboard interactivo no Streamlit](https://LINK-STREAMLIT-AQUI.streamlit.app)

---

## Contexto

Análise exploratória das **952 músicas mais ouvidas no Spotify em 2023**, combinando atributos musicais técnicos (BPM, danceability, energy…) com métricas reais de negócio em 4 plataformas: Spotify, Apple Music, Deezer e Shazam.

## Estrutura

| Secção | Conteúdo |
|--------|----------|
| **1. ETL** | Carregamento, diagnóstico de qualidade, limpeza e feature engineering |
| **2. EDA — Q1** | Quão desigual é a distribuição de streams? |
| **3. EDA — Q2** | Existe sazonalidade nos lançamentos musicais? |
| **4. EDA — Q3** | Músicas antigas ainda dominam os charts de 2023? |
| **5. EDA — Q4** | Colaborações geram mais streams do que solos? |
| **6. Sumário** | Insights e próximos passos |


## 0. Imports e configuração

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings("ignore")

# ── Paleta Spotify ─────────────────────────────────────────────────────────
GREEN  = "#1DB954"
CORAL  = "#E8563A"
AMBER  = "#F59B23"
PURPLE = "#9B72CF"
BLUE   = "#4FC3F7"
GRAY   = "#B3B3B3"
BG     = "#121212"
BG2    = "#1E1E1E"

ERA_COLORS = {
    "Pré-2000":  "#B3B3B3",
    "2000s":     "#9B72CF",
    "2010s":     "#4FC3F7",
    "2020–2021": "#1DB954",
    "2022–2023": "#E8563A",
}

# Template Plotly com identidade Spotify
TEMPLATE = go.layout.Template(
    layout=go.Layout(
        paper_bgcolor=BG,
        plot_bgcolor=BG2,
        font=dict(family="Arial", color=GRAY, size=12),
        title=dict(font=dict(color="white", size=15), x=0.01),
        xaxis=dict(gridcolor="#282828", linecolor="#282828", tickcolor="#535353", zerolinecolor="#282828"),
        yaxis=dict(gridcolor="#282828", linecolor="#282828", tickcolor="#535353", zerolinecolor="#282828"),
        colorway=[GREEN, CORAL, AMBER, PURPLE, BLUE, GRAY],
        legend=dict(bgcolor=BG2, bordercolor="#282828", borderwidth=1, font=dict(color=GRAY)),
        margin=dict(l=50, r=30, t=55, b=50),
        hoverlabel=dict(bgcolor="#282828", bordercolor="#535353", font=dict(color="white", size=12)),
    )
)

print("✅ Imports e paleta Spotify configurados")

---
## 1. ETL — Extracção, Transformação e Carregamento

### 1.1 Carregar dados

In [ ]:
df_raw = pd.read_csv(
    "spotify-2023.csv",
    encoding="utf-8",
    encoding_errors="replace",
)

print(f"Dimensões: {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas")
df_raw.head(3)

### 1.2 Diagnóstico de qualidade

In [ ]:
missing   = df_raw.isnull().sum()
empty_str = (df_raw == "").sum()
issues    = (missing + empty_str).rename("total")

print("Colunas com problemas de qualidade:\n")
display(
    issues[issues > 0]
    .reset_index()
    .rename(columns={"index": "coluna"})
    .assign(nulos=missing[issues > 0].values,
            vazios=empty_str[issues > 0].values)
)

**Issues identificados e decisões de tratamento:**

| Coluna | Problema | Decisão |
|--------|----------|---------|
| `streams` | 1 linha corrompida — dados de BPM/key entraram na coluna errada | Remover a linha |
| `key` | 95 valores em branco | Substituir por `'Unknown'` |
| `in_shazam_charts` | 50 valores nulos | Substituir por `0` (não entrou nos charts) |

### 1.3 Limpeza

In [ ]:
# Remover linha com streams corrompido
bad_mask = pd.to_numeric(df_raw["streams"], errors="coerce").isna()
print(f"Linha removida: '{df_raw.loc[bad_mask, 'track_name'].values[0]}'")

df = df_raw[~bad_mask].copy()
df["streams"] = pd.to_numeric(df["streams"])

# Converter colunas numéricas
num_cols = [
    "artist_count", "released_year", "released_month", "released_day",
    "in_spotify_playlists", "in_spotify_charts",
    "in_apple_playlists", "in_apple_charts",
    "in_deezer_playlists", "in_deezer_charts", "in_shazam_charts",
    "bpm", "danceability_%", "valence_%", "energy_%",
    "acousticness_%", "instrumentalness_%", "liveness_%", "speechiness_%",
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Tratar key e shazam
df["key"] = df["key"].replace("", np.nan).fillna("Unknown")
df["in_shazam_charts"] = df["in_shazam_charts"].fillna(0)

print(f"\n✅ Dataset limpo: {df.shape[0]} linhas × {df.shape[1]} colunas")

### 1.4 Feature engineering

In [ ]:
df["streams_M"] = df["streams"] / 1e6

df["era"] = pd.cut(
    df["released_year"],
    bins=[0, 1999, 2009, 2019, 2021, 2023],
    labels=["Pré-2000", "2000s", "2010s", "2020–2021", "2022–2023"],
).astype(str)

df["collab"] = df["artist_count"].apply(lambda x: "Colaboração" if x > 1 else "Solo")

print("Features criadas: streams_M, era, collab")
df[["track_name", "artist(s)_name", "released_year", "era", "collab", "streams_M"]].head(5)

### 1.5 Estatísticas descritivas

In [ ]:
desc = df["streams"].describe()

print("Streams — Estatísticas descritivas\n")
print(f"  Contagem : {int(desc['count']):,} músicas")
print(f"  Média    : {desc['mean']/1e6:>8.1f} M")
print(f"  Mediana  : {desc['50%']/1e6:>8.1f} M")
print(f"  Mínimo   : {desc['min']/1e6:>8.3f} M")
print(f"  Máximo   : {desc['max']/1e6:>8.1f} M")
print(f"  Std      : {desc['std']/1e6:>8.1f} M")
print()
print(f"⚠️  Média ({desc['mean']/1e6:.0f}M) >> Mediana ({desc['50%']/1e6:.0f}M)")
print("   → distribuição fortemente assimétrica à direita")

---
## 2. Q1 — Quão desigual é a distribuição de streams?

**Hipótese:** O mercado musical segue uma lei de potência — poucos artistas concentram a maioria dos streams.  
**Métricas:** histograma, distribuição log₁₀, Curva de Lorenz, Coeficiente de Gini.

In [ ]:
# Curva de Lorenz e Gini
s_sorted    = np.sort(df["streams"].values)
cum_streams = np.cumsum(s_sorted) / s_sorted.sum()
cum_pop     = np.arange(1, len(s_sorted) + 1) / len(s_sorted)
gini        = 1 - 2 * np.trapezoid(cum_streams, cum_pop)
idx_10      = int(0.9 * len(s_sorted))
share_top10 = (1 - cum_streams[idx_10]) * 100

print(f"Gini        = {gini:.3f}")
print(f"Top 10%     = {share_top10:.1f}% dos streams")
print(f"Média       = {df['streams_M'].mean():.0f}M")
print(f"Mediana     = {df['streams_M'].median():.0f}M")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Distribuição em log₁₀", "Curva de Lorenz"],
)

# Panel 1 — Histograma log10
log_s = np.log10(df["streams"].clip(lower=1))
fig.add_trace(
    go.Histogram(x=log_s, nbinsx=40, marker_color=GREEN, opacity=0.85,
                 hovertemplate="log₁₀: %{x:.2f}<br>Músicas: %{y}<extra></extra>",
                 name="Distribuição"),
    row=1, col=1
)
fig.add_vline(x=float(np.log10(df["streams"].median())),
              line_dash="dash", line_color=CORAL, line_width=2,
              annotation_text=f"Mediana {df['streams_M'].median():.0f}M",
              annotation_font_color=CORAL, row=1, col=1)
fig.add_vline(x=float(np.log10(df["streams"].mean())),
              line_dash="dot", line_color=AMBER, line_width=2,
              annotation_text=f"Média {df['streams_M'].mean():.0f}M",
              annotation_font_color=AMBER, annotation_position="top left", row=1, col=1)

# Panel 2 — Lorenz
fig.add_trace(
    go.Scatter(x=cum_pop*100, y=cum_streams*100,
               mode="lines", line=dict(color=GREEN, width=2.5),
               fill="tonexty", fillcolor="rgba(29,185,84,0.1)",
               name="Curva de Lorenz",
               hovertemplate="Top %{x:.1f}% músicas = %{y:.1f}% streams<extra></extra>"),
    row=1, col=2
)
fig.add_trace(
    go.Scatter(x=[0,100], y=[0,100], mode="lines",
               line=dict(color="#535353", width=1.2, dash="dash"),
               name="Igualdade perfeita"),
    row=1, col=2
)
fig.add_annotation(
    x=72, y=28, xref="x2", yref="y2",
    text=f"<b>Top 10% = {share_top10:.0f}% streams</b><br>Gini = {gini:.2f}",
    showarrow=True, arrowhead=2, ax=0, ay=-45,
    bgcolor=BG2, bordercolor=GREEN, borderwidth=1,
    font=dict(color=GREEN, size=12),
)

fig.update_layout(
    template=TEMPLATE, height=420,
    title_text="Q1 — Distribuição de streams",
    showlegend=False,
)
fig.update_xaxes(title_text="log₁₀(Streams)", row=1, col=1)
fig.update_yaxes(title_text="Nº de músicas", row=1, col=1)
fig.update_xaxes(title_text="% músicas (acumulado)", row=1, col=2)
fig.update_yaxes(title_text="% streams (acumulado)", row=1, col=2)

fig.show()

### 💡 Insights — Q1

| Métrica | Valor |
|---------|-------|
| Coeficiente de Gini | **0.53** |
| Top 10% das músicas | **37%** de todos os streams |
| Média | **514M** streams |
| Mediana | **291M** streams |

- A distribuição segue uma **lei de potência** — média quase o dobro da mediana, sinal de cauda longa extrema
- O **Gini de 0.53** é superior ao da desigualdade de rendimento em Portugal (~0.33)
- A transformação **log₁₀** aproxima os dados de uma normal — relevante para modelação futura
- **Implicação:** o algoritmo de playlists editoriais do Spotify é o principal driver de sucesso — sem promoção, a esmagadora maioria das músicas fica na cauda

---
## 3. Q2 — Existe sazonalidade nos lançamentos musicais?

**Hipótese:** A indústria concentra lançamentos no início do ano e antes do Verão — mas o mês afecta o número de streams?  
**Nota:** Análise focada em 2022–2023 (anos com dados suficientemente densos).

In [ ]:
month_names = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]

df_recent = df[df["released_year"] >= 2022].copy()
m_count   = df_recent.groupby("released_month").size().reindex(range(1,13), fill_value=0)
m_streams = df_recent.groupby("released_month")["streams_M"].median().reindex(range(1,13), fill_value=0)

print(f"Mês com mais lançamentos  : {month_names[m_count.idxmax()-1]} ({m_count.max()} músicas)")
print(f"Mês com menos lançamentos : {month_names[m_count.idxmin()-1]} ({m_count.min()} músicas)")
print(f"Mês com mais streams (med): {month_names[m_streams.idxmax()-1]} ({m_streams.max():.0f}M)")

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Volume de lançamentos por mês", "Streams mediana por mês"],
)

# Volume
colors_vol = [CORAL if v == m_count.max() else GREEN for v in m_count.values]
fig.add_trace(
    go.Bar(x=month_names, y=m_count.values, marker_color=colors_vol, opacity=0.9,
           text=m_count.values, textposition="outside",
           textfont=dict(color="white", size=11),
           hovertemplate="%{x}: <b>%{y} músicas</b><extra></extra>"),
    row=1, col=1
)
fig.add_hline(y=m_count.mean(), line_dash="dash", line_color="#535353",
              annotation_text=f"Média {m_count.mean():.0f}", row=1, col=1)

# Streams mediana
colors_str = [AMBER if v == m_streams.max() else GREEN for v in m_streams.values]
fig.add_trace(
    go.Bar(x=month_names, y=m_streams.values, marker_color=colors_str, opacity=0.9,
           text=[f"{v:.0f}M" for v in m_streams.values], textposition="outside",
           textfont=dict(color="white", size=11),
           hovertemplate="%{x}: mediana <b>%{y:.0f}M</b><extra></extra>"),
    row=1, col=2
)

fig.update_layout(
    template=TEMPLATE, height=420,
    title_text="Q2 — Sazonalidade nos lançamentos (2022–2023)",
    showlegend=False,
)
fig.update_yaxes(title_text="Nº de músicas", row=1, col=1)
fig.update_yaxes(title_text="Streams mediana (M)", row=1, col=2)

fig.show()

### 💡 Insights — Q2

- **Janeiro** lidera em volume (134 músicas) — a indústria abre o ano forte
- **Agosto** é o mês mais calmo (46 músicas) — férias da indústria
- Músicas lançadas em **Outubro** têm a maior mediana de streams — menos concorrência = mais visibilidade por música
- **Implicação:** há um trade-off claro entre lançar na "época alta" (volume) vs. lançar quando há menos ruído competitivo (streams/música)

---
## 4. Q3 — Músicas antigas ainda dominam os charts de 2023?

**Hipótese:** Catálogos antigos sobrevivem nos charts porque são clássicos consolidados — mas isso cria um viés de sobrevivência.  
**Metodologia:** Comparação de distribuição de streams por era de lançamento.

In [ ]:
era_order = ["Pré-2000", "2000s", "2010s", "2020–2021", "2022–2023"]

era_summary = (
    df.groupby("era")["streams_M"]
    .agg(["count", "median", "mean", "max"])
    .reindex(era_order)
    .rename(columns={"count":"Músicas","median":"Mediana (M)","mean":"Média (M)","max":"Máx (M)"})
    .round(0)
)
display(era_summary)

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Nº de músicas por era", "Distribuição de streams por era (boxplot)"],
)

era_counts = df["era"].value_counts().reindex(era_order, fill_value=0)

# Barras contagem
fig.add_trace(
    go.Bar(
        x=era_order,
        y=era_counts.values,
        marker_color=[ERA_COLORS[e] for e in era_order],
        opacity=0.9,
        text=era_counts.values, textposition="outside",
        textfont=dict(color="white", size=11),
        hovertemplate="%{x}: <b>%{y} músicas</b><extra></extra>",
    ),
    row=1, col=1,
)

# Boxplot por era
for era in era_order:
    sub = df[df["era"] == era]
    if len(sub) == 0:
        continue
    color = ERA_COLORS[era]
    r, g, b = int(color[1:3],16), int(color[3:5],16), int(color[5:7],16)
    fig.add_trace(
        go.Box(
            y=sub["streams_M"], name=era,
            marker_color=color, line_color=color,
            fillcolor=f"rgba({r},{g},{b},0.15)",
            boxmean=True,
            hovertemplate=f"<b>{era}</b><br>%{{y:.0f}}M<extra></extra>",
        ),
        row=1, col=2,
    )

fig.update_layout(
    template=TEMPLATE, height=440,
    title_text="Q3 — Músicas antigas vs. recentes nos charts de 2023",
    showlegend=False,
)
fig.update_yaxes(title_text="Nº de músicas", row=1, col=1)
fig.update_yaxes(title_text="Streams (M)", row=1, col=2)

fig.show()

### 💡 Insights — Q3

| Era | n | Mediana streams |
|-----|---|----------------|
| Pré-2000 | 48 | 622M |
| **2000s** | 20 | **1.229M** ← maior |
| 2010s | 151 | 984M |
| 2020–2021 | 156 | 564M |
| 2022–2023 | 577 | 179M |

> ⚠️ **Viés de sobrevivência:** as únicas músicas antigas que chegam aos charts de 2023 são os maiores clássicos de sempre. As milhares de músicas "normais" dessa era simplesmente não estão neste dataset — o que infla artificialmente a mediana das eras antigas.

- **Implicação:** o catálogo histórico é um activo de altíssimo valor para editoras — gera streams passivos sem custo de marketing

---
## 5. Q4 — Colaborações geram mais streams do que solos?

**Hipótese:** Colaborações combinam fandoms e deveriam ter mais alcance.  
**Metodologia:** Comparação de medianas Solo vs. Colaboração; análise por número de artistas.

In [ ]:
solo_med   = df[df["collab"] == "Solo"]["streams_M"].median()
collab_med = df[df["collab"] == "Colaboração"]["streams_M"].median()
diff_pct   = (solo_med / collab_med - 1) * 100

print(f"Solo mediana        : {solo_med:.0f}M")
print(f"Colaboração mediana : {collab_med:.0f}M")
print(f"Diferença           : Solo +{diff_pct:.0f}% vs. Colaboração")

In [ ]:
df["artist_count_capped"] = df["artist_count"].clip(upper=5)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Volume no dataset", "Streams mediana", "Mediana por nº de artistas"],
)

# Volume
counts = df["collab"].value_counts().reindex(["Solo","Colaboração"], fill_value=0)
fig.add_trace(
    go.Bar(x=counts.index.tolist(), y=counts.values,
           marker_color=[GREEN, CORAL], opacity=0.9,
           text=[f"{v}<br>({v/len(df)*100:.0f}%)" for v in counts.values],
           textposition="outside", textfont=dict(color="white", size=11),
           hovertemplate="%{x}: <b>%{y}</b><extra></extra>"),
    row=1, col=1,
)

# Mediana
medians = df.groupby("collab")["streams_M"].median().reindex(["Solo","Colaboração"])
fig.add_trace(
    go.Bar(x=medians.index.tolist(), y=medians.values,
           marker_color=[GREEN, CORAL], opacity=0.9,
           text=[f"{v:.0f}M" for v in medians.values],
           textposition="outside", textfont=dict(color="white", size=12),
           hovertemplate="%{x}: mediana <b>%{y:.0f}M</b><extra></extra>"),
    row=1, col=2,
)

# Por nº de artistas
by_n     = df.groupby("artist_count_capped")["streams_M"].median()
x_labels = {1:"1 (solo)",2:"2",3:"3",4:"4",5:"5+"}
colors_n = [GREEN, BLUE, AMBER, CORAL, PURPLE]
fig.add_trace(
    go.Bar(
        x=[x_labels.get(int(k), str(k)) for k in by_n.index],
        y=by_n.values,
        marker_color=colors_n[:len(by_n)], opacity=0.9,
        text=[f"{v:.0f}M" for v in by_n.values],
        textposition="outside", textfont=dict(color="white", size=11),
        hovertemplate="Nº artistas %{x}: <b>%{y:.0f}M</b><extra></extra>",
    ),
    row=1, col=3,
)

fig.update_layout(
    template=TEMPLATE, height=420,
    title_text="Q4 — Colaborações vs. Solos",
    showlegend=False,
)
fig.update_yaxes(title_text="Nº de músicas", row=1, col=1)
fig.update_yaxes(title_text="Streams mediana (M)", row=1, col=2)
fig.update_xaxes(title_text="Nº de artistas", row=1, col=3)

fig.show()

### 💡 Insights — Q4

- **Resultado contra-intuitivo:** solos têm mediana **41% superior** às colaborações (334M vs. 237M)
- **Hipótese explicativa:** artistas com bases de fãs consolidadas (Taylor Swift, Bad Bunny, The Weeknd) lançam maioritariamente solos; colaborações são frequentemente usadas por artistas em crescimento para ganhar exposição
- **O factor determinante é a base de fãs do artista principal**, não o número de artistas na faixa
- **Limitação:** este dataset só inclui músicas *já populares* — não podemos extrapolar para artistas em início de carreira

---
## 6. Sumário Executivo

| # | Pergunta | Resposta-chave | Implicação de negócio |
|---|----------|---------------|----------------------|
| **Q1** | Distribuição de streams | Gini 0.53 · top 10% = 37% streams | Mercado concentrado — algoritmo é decisivo |
| **Q2** | Sazonalidade | Jan/Mai = mais lançamentos; Out = mais streams/música | Trade-off entre volume e visibilidade |
| **Q3** | Músicas antigas | 2000s têm mediana 6× maior que 2022–2023 | Viés de sobrevivência; catálogo = activo passivo |
| **Q4** | Colaborações | Solos com mediana 41% superior | Base de fãs do artista principal é o factor decisivo |

---

## 7. Próximos passos

- [ ] **Correlações:** que atributos musicais (BPM, energy, danceability) mais se correlacionam com streams?
- [ ] **Teste de hipóteses:** modo Maior vs. Menor tem impacto estatisticamente significativo?
- [ ] **Clustering:** que perfis de músicas emergem sem usar o género como label?
- [ ] **Modelo preditivo:** é possível prever streams a partir de atributos musicais?

---

> 🎵 Para a versão interactiva com filtros em tempo real → **[Dashboard Streamlit](https://LINK-STREAMLIT-AQUI.streamlit.app)**
